<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
Erste Agenten mit LangChain
</b></font> </br></p>

---

**Beitrag zum Leitprojekt:** Hier trifft der Meeting- & Research-Briefing-Agent seine ersten eigenständigen Entscheidungen — welches Tool zu einer Anfrage passt und wann eine direkte Antwort ausreicht. Das ist der Kern von **Planen**: Der Agent wählt den nächsten sinnvollen Schritt, bevor er handelt.

In [ ]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith erst im Trace-Abschnitt verwenden.
import os
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGSMITH_PROJECT"] = "M02-Erste-Agenten"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import setup_api_keys, mprint, show_trace
from genai_lib.model_config import WORKER

setup_api_keys(["OPENAI_API_KEY", "LANGSMITH_API_KEY"], create_globals=False)


✓ OPENAI_API_KEY erfolgreich gesetzt
✓ LANGSMITH_API_KEY erfolgreich gesetzt

Python Version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]

Installierte LangChain- und LangGraph-Bibliotheken:
langchain                                1.4.3
langchain-anthropic                      1.7.5
langchain-chroma                         1.1.0
langchain-classic                        1.0.8
langchain-community                      0.4.2
langchain-core                           1.6.6
langchain-google-genai                   4.4.0
langchain-ollama                         1.1.0
langchain-openai                         1.6.7
langchain-protocol                       0.0.19
langchain-text-splitters                 1.1.3
langgraph                                1.2.11
langgraph-checkpoint                     4.2.0
langgraph-checkpoint-sqlite              3.1.1
langgraph-prebuilt                       1.1.0
langgraph-sdk                            0.4.4

IP-Adresse: 34.26.246.217
Hostname: 217.246.26.

# 1 | Übersicht
---

In M01 wurden Werkzeuge als normale Python-Funktionen definiert und direkt getestet. Jetzt lernt der Meeting- & Research-Briefing-Agent, selbst eines dieser Werkzeuge auszuwählen.

Der Weg durch M02:
1. Ein Chatmodell vorbereiten.
2. Eine Python-Funktion mit `@tool` für den Agenten bereitstellen.
3. Einen Agenten mit einem Tool ausführen.
4. Ein zweites Tool ergänzen und die Entscheidungen beobachten.
5. Den Nachrichtenverlauf und einen LangSmith-Trace lesen.

Für den ersten Agenten reichen **Modell, Tools und System-Prompt**. Prompt-Templates folgen in M03, strukturierte Ausgaben in M04 und Chains in M06.


<p><font color='darkblue' size="4">
 <b>✨ Viz</b>
</font></p>

- [KI-Agent](https://editor.p5js.org/ralf.bendig.rb/full/u3Ee0jtFo)

# 2 | LangChain: die Bausteine für den ersten Agenten
---


LangChain verbindet ein Chatmodell mit Python-Funktionen und verwaltet den Ablauf eines Agenten. Für dieses Notebook genügen vier Begriffe:

| Begriff | Bedeutung hier |
|---|---|
| **Chatmodell** | Erzeugt Antworten und kann einen Tool-Aufruf vorschlagen. |
| **Nachricht** | Enthält eine Rolle und einen Text, zum Beispiel eine Nutzerfrage. |
| **Tool** | Eine beschriebene Python-Funktion, die der Agent ausführen kann. |
| **Agent** | Übergibt Nachrichten an das Modell, führt gewählte Tools aus und liefert eine Antwort. |

Das Modell wird einmal initialisiert. Erst ein späterer Aufruf mit `.invoke(...)` sendet eine Anfrage.


In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(WORKER)


# 3 | Von der Python-Funktion zum Tool
---


M01 hat Funktionen direkt aufgerufen. Mit `@tool` erhält eine Python-Funktion einen Namen, eine Beschreibung aus dem Docstring und ein Argument-Schema aus den Typannotationen. Der Agent kann sie dadurch einem Chatmodell anbieten und bei Bedarf ausführen.

Wir beginnen mit **einem** Tool und testen es erst ohne Agentenaufruf.


In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_agent


In [ ]:
@tool
def begriffe_extrahieren(text: str) -> str:
    """Extrahiert bis zu fünf Schlüsselbegriffe aus einem kurzen Fachtext."""
    stopwoerter = {"und", "oder", "der", "die", "das", "ein", "eine", "mit", "für", "von", "im", "in"}
    woerter = [wort.strip(".,:;!?()[]").lower() for wort in text.split()]
    kandidaten = []
    for wort in woerter:
        if len(wort) >= 5 and wort not in stopwoerter and wort not in kandidaten:
            kandidaten.append(wort)
    return ", ".join(kandidaten[:5]) or "Keine Schlüsselbegriffe gefunden."


In [ ]:
mprint(f"**Tool-Name:** `{begriffe_extrahieren.name}`")
mprint(f"**Beschreibung:** {begriffe_extrahieren.description}")
mprint(f"**Parameter:** `{begriffe_extrahieren.args}`")
mprint(f"**Direkter Test:** {begriffe_extrahieren.invoke({'text': 'RAG verbindet Retrieval und Antworten.'})}")


# 4 | Erster Agent mit einem Tool
---


`create_agent()` verbindet drei Teile:

| Parameter | Aufgabe |
|---|---|
| `model` | Das zuvor initialisierte Chatmodell. |
| `tools` | Die Funktionen, die der Agent verwenden darf. |
| `system_prompt` | Die Regel, wann und wie der Agent antworten soll. |

Der System-Prompt bleibt zunächst kurz. Die gezielte Prompt-Steuerung folgt in M03.


In [ ]:
system_prompt = (
    "Du unterstützt Meeting- und Research-Briefings. "
    "Nutze das Tool, wenn du Schlüsselbegriffe aus einem Fachtext bestimmen sollst. "
    "Antworte knapp und erfinde keine Rechercheergebnisse."
)

agent = create_agent(
    model=llm,
    tools=[begriffe_extrahieren],
    system_prompt=system_prompt,
)

mprint(f"**Agent bereit:** `{begriffe_extrahieren.name}` ist verfügbar.")


**Agent bereit** mit folgenden Tools:

- `begriffe_extrahieren`: Extrahiert bis zu fünf Schlüsselbegriffe aus einem kurzen Fachtext.

- `korpus_check`: Prüft grob, ob ein Thema im Kurs-Wissenskorpus erwartet wird.

Die erste Anfrage verlangt genau die Fähigkeit des verfügbaren Tools. Die Eingabe besteht aus einer Liste von Nachrichten; `role: user` kennzeichnet die Nutzerfrage. Das Ergebnis enthält die Antwort und den Nachrichtenverlauf.


In [ ]:
ergebnis = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Extrahiere Schlüsselbegriffe aus: Retrieval Augmented Generation verbindet Suche, Kontext und Antwortgenerierung.",
    }]
})

mprint("## Agent-Antwort")
mprint(ergebnis["messages"][-1].content)


## Agent-Antwort

RAG ist im Wissenskorpus vertreten.

Schlüsselbegriffe:
- retrieval
- augmented
- generation
- verbindet
- suche

In [ ]:
mprint("## Tool-Auswahl im ersten Lauf")
tool_aufrufe = [
    aufruf["name"]
    for nachricht in ergebnis["messages"]
    for aufruf in getattr(nachricht, "tool_calls", [])
]
mprint(f"**Aufgerufene Tools:** `{tool_aufrufe}`")


## Verwendete Tool-Calls

- `korpus_check({'thema': 'RAG'})`

- `begriffe_extrahieren({'text': 'Retrieval Augmented Generation verbindet Suche, Kontext und Antwortgenerierung.'})`

# 5 | Zweites Tool und Verhalten testen
---


Jetzt kommt eine zweite Fähigkeit hinzu: eine einfache Prüfung der Korpusabdeckung. Danach erstellen wir den Agenten mit **beiden** Tools und vergleichen Anfragen, die ein Tool, beide Tools oder keines benötigen.


In [ ]:
@tool
def korpus_check(thema: str) -> str:
    """Prüft grob, ob ein Thema im Kurs-Wissenskorpus erwartet wird."""
    bekannte_themen = {
        "rag": "RAG ist im Wissenskorpus vertreten.",
        "retrieval": "Retrieval ist im Wissenskorpus vertreten.",
        "evaluation": "Evaluation ist im Wissenskorpus vertreten.",
        "agent": "Agenten sind im Wissenskorpus vertreten.",
        "langgraph": "LangGraph ist im Wissenskorpus voraussichtlich nur indirekt vertreten.",
    }
    thema_normalisiert = thema.lower()
    for schluessel, antwort in bekannte_themen.items():
        if schluessel in thema_normalisiert:
            return antwort
    return "Keine sichere Korpusabdeckung erkennbar."


In [ ]:
agent = create_agent(
    model=llm,
    tools=[begriffe_extrahieren, korpus_check],
    system_prompt=system_prompt,
)

mprint("**Agent bereit** mit `begriffe_extrahieren` und `korpus_check`.")


In [ ]:
test_fragen = [
    ("Tool erwartet", "Welche Schlüsselbegriffe stecken in: Agentic RAG kombiniert Planung, Retrieval und Evaluation."),
    ("Tool erwartet", "Ist Evaluation im Wissenskorpus vertreten?"),
    ("Kein Tool", "Erkläre kurz, was ein KI-Agent ist."),
    ("Beide Tools", "Prüfe, ob RAG im Korpus vorkommt, und extrahiere Begriffe aus: RAG nutzt Retrieval für fundierte Antworten."),
]

In [ ]:
for erwartung, frage in test_fragen:
    result = agent.invoke({"messages": [{"role": "user", "content": frage}]})
    tools_genutzt = [
        aufruf["name"]
        for nachricht in result["messages"]
        for aufruf in getattr(nachricht, "tool_calls", [])
    ]

    mprint(f"
---
**Frage [{erwartung}]:** {frage}")
    mprint(f"**Antwort:** {result['messages'][-1].content}")
    mprint(f"**Tools genutzt:** `{tools_genutzt}`")



---
**Frage [Tool erwartet]:** Welche Schlüsselbegriffe stecken in: Agentic RAG kombiniert Planung, Retrieval und Evaluation.

**Antwort:** Schlüsselbegriffe:
- Agentic RAG
- Planung
- Retrieval
- Evaluation
- kombiniert

**Tools genutzt:** `begriffe_extrahieren`


---
**Frage [Tool erwartet]:** Ist Evaluation im Wissenskorpus vertreten?

**Antwort:** Ja, **Evaluation** ist im Wissenskorpus vertreten.

...

**Antwort:** Ein KI-Agent ist ein System, das **selbstständig ein Ziel verfolgt**, dabei **seine Umgebung wahrnimmt**, **Entscheidungen trifft** und **Handlungen ausführt**, um das Ziel zu erreichen.

Kurz gesagt:  
**Wahrnehmen → Denken/Planen → Handeln → Prüfen → anpassen**

Beispiel: Ein Agent kann eine Aufgabe wie „Finde die beste Zugverbindung und buche sie“ nicht nur beantworten, sondern auch Schritte planen und ausführen.

**Kein Tool genutzt** - direkte LLM-Antwort


---
**Frage [Beide Tools]:** Prüfe, ob RAG im Korpus vorkommt, und extrahiere Begriffe aus: RAG nutzt Retrieval für fundierte Antworten.

**Antwort:** RAG kommt im Korpus vor.

Extrahierte Begriffe:
- nutzt
- Retrieval
- fundierte
- Antworten

**Tools genutzt:** `korpus_check`, `begriffe_extrahieren`

# 6 | Nachrichten und Entscheidungen verstehen
---

Im gespeicherten ersten Lauf `ergebnis` stehen die einzelnen Schritte unter `messages`:

- **`HumanMessage`**: die Nutzerfrage.
- **`AIMessage`**: eine Tool-Entscheidung oder die abschließende Antwort des Modells.
- **`ToolMessage`**: das Ergebnis eines ausgeführten Tools.

Bei einer festen Python-Abfolge wären die Aufrufe vorher festgelegt. Hier entscheidet der Agent anhand der Anfrage, welches Tool er nutzt. Chains als feste LangChain-Abläufe werden in M06 behandelt.


In [ ]:
nachrichten = ergebnis["messages"]
mprint(f"**Nachrichten im ersten Lauf:** {len(nachrichten)}")
for nachricht in nachrichten:
    typ = type(nachricht).__name__
    tool_namen = [aufruf["name"] for aufruf in getattr(nachricht, "tool_calls", [])]
    inhalt = str(nachricht.content)[:160] or "(kein Text)"
    mprint(f"- **{typ}**: {inhalt} – Tool-Aufrufe: `{tool_namen}`")


Schlüssel im Ergebnis: ['messages']
Anzahl der Nachrichten im Verlauf: 5


# 7 | LangSmith: Agent-Trace
---


Ein Trace zeigt die sichtbaren Schritte eines Agentenlaufs: Modellentscheidung, Tool-Aufruf mit Parametern, Tool-Ergebnis und Antwort. Er ergänzt den Nachrichtenverlauf um eine gut lesbare Ablaufansicht.

Für diesen Referenzlauf wird LangSmith einmal gezielt aktiviert. Versteckte Modellgedanken sind dabei nicht sichtbar.


In [ ]:
from langchain_core.tracers.context import tracing_v2_enabled

run_cfg = {
    "run_name": "M02_Kap7_AgentTrace",
    "tags": ["M02", "agent", "langsmith", "meeting-research-briefing"],
}

with tracing_v2_enabled(project_name="M02-Erste-Agenten", tags=run_cfg["tags"]):
    trace_result = agent.with_config(**run_cfg).invoke({
        "messages": [{
            "role": "user",
            "content": "Ist Retrieval im Wissenskorpus vertreten, und welche Schlüsselbegriffe enthält: Retrieval verbessert Antworten durch dokumentnahen Kontext?",
        }]
    })

mprint("## Trace-Ergebnis")
mprint(trace_result["messages"][-1].content)


## Trace-Ergebnis

---

Ja, **Retrieval** ist im Wissenskorpus vertreten.

**Schlüsselbegriffe** aus dem Text:
- Retrieval
- verbessert
- Antworten
- durch
- dokumentnahen

Wenn du möchtest, kann ich die Begriffe auch noch **sinnvoll bereinigen/normalisieren**, z. B. zu: **Retrieval, Antworten, dokumentnaher Kontext**.

In [ ]:
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M02-Erste-Agenten", limit=3, show_steps=True)

## LangSmith Trace — `M02-Erste-Agenten`

| Run | Status | Dauer | Child-Runs |
|-----|--------|-------|------------|
| `M02_Kap5_AgentTrace` | ❌ pending | — | 0 |
| `M02_Kap5_AgentTrace` | ✅ success | 1.6s | 0 |


### Steps — letzter Run: `M02_Kap5_AgentTrace`

| # | Typ | Name | Status | Dauer |
|---|-----|------|--------|-------|
| 1 | `chain` | `model` | ✅ | 1.6s |
| 2 | `chain` | `tools` | ✅ | 0.0s |
| ↳ | `tool` | `begriffe_extrahieren` | ✅ | 0.0s |
| 3 | `chain` | `tools` | ✅ | 0.0s |
| ↳ | `tool` | `korpus_check` | ✅ | 0.0s |
| 4 | `chain` | `model` | ✅ | 1.0s |

## Übergang zu M03
---

M02 hat gezeigt, wie ein Agent zwischen direkter Antwort und Tool-Aufruf entscheidet. M03 untersucht als Nächstes, wie System-Prompts diese Entscheidungen und die Antwortform gezielt steuern.


# A | Aufgaben
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabenstellungen unten bieten Anregungen; alternative Herausforderungen sind möglich.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs darf und soll generative KI auch als Unterstützung beim Lernen und Entwickeln genutzt werden. Geeignet ist sie zum Beispiel, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.




<p><font color='black' size="5">
Erster eigener Agent
</font></p>

Einen ersten Agenten für den Meeting-Briefing-Agent erstellen. Die Aufgabe baut auf M01 auf: Ein Tool wird definiert, in `create_agent()` eingebunden und mit einer Anfrage getestet.

**Grundlagen**
1. Ein eigenes Briefing-Tool mit dem `@tool`-Decorator definieren, zum Beispiel `paper_signal_extrahieren(text: str)`.
2. Einen Agenten mit `create_agent()` und diesem Tool erstellen.
3. Eine einfache Testfrage stellen, bei der das Tool nützlich ist, und die Ausgabe zeigen.

Den Agenten in der Variable `mein_agent` speichern.

**✅ Erledigt wenn:** `mein_agent.invoke({"messages": [...]})` gibt eine Antwort zurück und der Selbstcheck läuft fehlerfrei durch.


In [ ]:
from genai_lib.model_config import WORKER
# Grundlagen: Ein Tool + Agent
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import tool

# ...

**Aufbau**
1. Zwei Briefing-Tools definieren: eines für Textsignale, eines für Korpusabdeckung.
2. Einen System-Prompt schreiben, der die Rolle des Agenten klar beschreibt.
3. Den Agenten mit beiden Tools und dem eigenen System-Prompt erstellen.
4. Drei Testfragen stellen - davon mindestens 1 Frage, bei der kein Tool nötig ist.
5. Kurz notieren, wann Tools genutzt werden und wann nicht.

Agenten in `mein_agent`, Tools in `meine_tools` speichern.

**✅ Erledigt wenn:** `meine_tools` enthält mindestens zwei Tools und der Agent beantwortet drei Testfragen.


In [ ]:
# Aufbau: Zwei Tools + System-Prompt + drei Testfragen
@tool
def begriffe_zaehlen(text: str) -> str:
    " ... "
    pass

@tool
def korpus_thema_checken(thema: str) -> str:
    " ... "
    pass

# ...

**Vertiefung**
1. Ein drittes Briefing-Tool hinzufügen.
2. 2 Anfragen stellen, bei denen der Agent selbst entscheidet, welches der 3 Tools passt.
3. Den System-Prompt anpassen, zum Beispiel strenger oder ausführlicher, und in 2-3 Sätzen beschreiben, wie sich das Verhalten ändert.
4. LangSmith-Tracing aktivieren und für einen Agenten-Lauf beschreiben, welche ReAct-Schritte im Trace sichtbar sind.

Die Beobachtung zum System-Prompt in `prompt_beobachtung` (String) speichern.

**✅ Erledigt wenn:** `prompt_beobachtung` enthält eine Beobachtung mit mindestens 20 Zeichen und der Vertiefungsagent antwortet auf zwei Testfragen.


In [ ]:
# Vertiefung: Drittes Tool + LangSmith
@tool
def out_of_corpus_hinweis(frage: str) -> str:
    " ... "



# B | Dokumente zum Weiterlesen
---

Ergänzende Artikel aus der Kurs-Dokumentation:

- [Einsteiger LangChain](https://ralf-42.github.io/Agenten/05-frameworks/einsteiger-langchain.html)
- [LangChain](https://ralf-42.github.io/Agenten/05-frameworks/langchain.html)
- [Agenten-Architekturen](https://ralf-42.github.io/Agenten/04-agenten-implementierung/entwurf/agent-architekturen.html)
- [Tool Use & Function Calling](https://ralf-42.github.io/Agenten/04-agenten-implementierung/entwurf/tool-use-function-calling.html)

